# 下载 Zhuang-ABCA-1

147 张冠状切片，1122 基因。优先用于与本地 ABCA-2 的跨动物验证。

[官方数据说明](https://alleninstitute.github.io/abc_atlas_access/descriptions/Zhuang-ABCA-1.html) · [官方获取教程](https://alleninstitute.github.io/abc_atlas_access/notebooks/zhuang_merfish_tutorial.html)

使用官方固定版 `20241115` manifest 解析真实路径、文件大小和 MD5。默认下载 raw 表达；需要官方 log2 表达时修改配置。

## 1. 环境

Python ≥3.11。下面安装到当前 notebook 内核；已有依赖可跳过此单元格。
下载与转换共用同目录的 `download_utils.py`、`h5ad_utils.py`，无需先运行其他 notebook。
官方文件保存在 `official/`，清单在 `metadata/`，H5AD 在 `h5ad/`。
支持 HTTP 断点续传；有官方校验值时核验校验值，否则验证传输长度。重复运行会检查并跳过已完成文件。

In [ ]:
%pip install -q "anndata>=0.11,<0.13" pandas scipy h5py pyarrow


## 2. 路径与参数

In [ ]:
from pathlib import Path
import sys
import pandas as pd

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "download").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 DeepSpatial 项目目录或 data/download 运行。")
sys.path.insert(0, str(PROJECT_ROOT / "data" / "download"))
from download_utils import download_records, save_json

DATA_DIR = PROJECT_ROOT / "data" / "Zhuang-ABCA-1"
DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_H5AD = False  # 可选：改为 True，在官方文件之外增加 H5AD；不修改原文件。
EXPRESSION_FORMATS = ("raw",)  # 可改为 ("raw", "log2")
EXPORT_FORMAT = "raw"
print("数据目录：", DATA_DIR)


## 3. 获取并查看官方文件清单

包含表达、带分类的细胞元数据、基因、CCF；独立脑数据另取 reconstructed coordinates。解剖对应表与分类层级随本数据集保存。

In [ ]:
from download_utils import allen_records

records = allen_records("Zhuang-ABCA-1", DATA_DIR, formats=EXPRESSION_FORMATS)
display(pd.DataFrame(records)[["role", "path", "size"]])
print(f"计划下载 {sum(r['size'] for r in records) / 1024**3:.2f} GiB（不含可选 H5AD）")


## 4. 下载并校验

In [ ]:
download_records(records, DATA_DIR)


## 5. 可选：按切片导出 H5AD

设置 `EXPORT_H5AD=True` 后运行。输出 `h5ad/raw/sec_XXX.h5ad`；保留原标签、表达处理状态和原坐标，不进行归一化或毫米/微米换算。
按 cell_label/gene_identifier 精确连接，保留没有 CCF 的细胞；CCF 坐标独立存放，不替代原切片坐标。
全脑元数据需要较大内存，表达矩阵按 backed 模式逐切片读取。部分前端 Z=0 的切片仍保留，导出不自动修订深度。

In [ ]:
if EXPORT_H5AD:
    from h5ad_utils import export_allen
    assert EXPORT_FORMAT in EXPRESSION_FORMATS
    h5ad_files = export_allen(DATA_DIR, records, expression_format=EXPORT_FORMAT)
    print(f"完成 {len(h5ad_files)} 张切片，索引见 metadata/h5ad_sections.csv")
else:
    print("官方数据已保留；需要 H5AD 时将 EXPORT_H5AD 改为 True 并运行本单元格。")
